# Training visualization

End-to-end view of every continued-pretraining trial that
`scripts/train_pipeline.py` has produced — per-epoch loss/metric
curves, learning-rate schedule, cross-trial comparisons,
convergence diagnostics, and resource-usage trade-offs.

## Where the data comes from

Two artefacts written by the training pipeline:

* **Per-epoch CSV** — one file per trial under
  `output/training/epochs/<track>/<descriptive_name>.csv` with the
  schema `(epoch, train_loss, lr, metric_name, train_metric,
  test_metric, epoch_time_sec, elapsed_sec)`.
* **Run manifest CSV** — `output/training/manifests/<run_name>_<track>.csv`,
  one row per trial: hyperparameters, status (OK / FAIL), total
  walltime, the path of the final-epoch checkpoint.

All loading and plotting code lives in
`src/utils/training_viz.py`; this notebook contains only function
calls so the notebook stays scannable and the logic stays testable.

## What a 'trial' is

One trial = one `(base_checkpoint × learning_rate × use_lora)` tuple
from `cfg.tunable` in `config/train.yaml`. The full grid is the
Cartesian product, so a four-base / four-LR / {no-LoRA, LoRA}
sweep is 32 trials per track. SLURM array indices map 1-to-1 onto
trial indices.

## Direction of improvement

Higher is better for PD (`roc_auc`); lower is better for LGD
(`rmse`). Every helper in this notebook is direction-aware — `best`
and `sort_values` calls flip sign automatically.

## Sections

1. Setup and what's on disk
2. Per-trial dashboards (one model at a time)
3. Cross-trial loss & metric curves
4. Final-metric comparisons by hyperparameter
5. Time / accuracy trade-off
6. Convergence diagnostics
7. Failures and leaderboard

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
REPO = Path(os.getcwd()).resolve()
while not (REPO / 'src' / 'utils' / 'training_viz.py').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import pandas as pd
from src.utils.figures import open_figure_sink
from src.utils.training_viz import (
    load_run_manifest, load_epoch_history, load_all_epoch_histories,
    training_overview, trial_leaderboard, failed_trials,
    plot_loss_curve, plot_lr_schedule, plot_metric_curves,
    plot_epoch_time, plot_trial_dashboard,
    plot_loss_overlay, plot_metric_overlay, plot_overfitting_diagnostic,
    plot_final_metric_bar, plot_lr_effect, plot_lora_effect,
    plot_metric_heatmap, plot_base_ranking,
    plot_pareto_time_vs_metric, plot_epoch_time_overlay,
    plot_convergence_speed,
)
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
TRACK = 'pd'    # toggle to 'lgd' to view the regressor sweep
sink = open_figure_sink('1.0_training_visualization')


## 1 · What's on disk

Manifest tells us which trials were attempted and their outcome;
the per-epoch CSV count tells us how many of those produced any
training history.

In [ ]:
manifest = load_run_manifest(TRACK)
manifest

In [ ]:
overview = training_overview(TRACK)
overview

## 2 · Per-trial dashboard

Pick one trial (any descriptive_name from the leaderboard). The
dashboard gives a 2×2 view of `loss`, `lr`, `train/test metric`,
and `epoch wall-clock` — useful for spotting one slow chunk, an
exploding loss, or a metric that's still climbing at the cliff.

In [ ]:
# Default: the best-scoring trial in the overview. Override with any
# trial_name string to inspect that one instead.
FOCUS_TRIAL = (
    overview.sort_values('best_test_metric', ascending=(TRACK == 'lgd'))['trial_name'].iloc[0]
    if not overview.empty else 'creditpfn_pd_tabpfn-v3-classifier-v3_default_lr5e-05_seed42'
)
FOCUS_TRIAL

In [ ]:
sink.save(plot_trial_dashboard(FOCUS_TRIAL, TRACK), '01_trial_dashboard')

In [ ]:
sink.save(plot_loss_curve(FOCUS_TRIAL, TRACK), '02_loss_curve')

In [ ]:
sink.save(plot_lr_schedule(FOCUS_TRIAL, TRACK), '03_lr_schedule')

In [ ]:
sink.save(plot_metric_curves(FOCUS_TRIAL, TRACK), '04_metric_curves')

In [ ]:
sink.save(plot_epoch_time(FOCUS_TRIAL, TRACK), '05_epoch_time')

## 3 · Cross-trial overlays

Every trial on one axes. Colour = base checkpoint;
solid = no-LoRA, dashed = LoRA. Use these to spot regimes:

* If LoRA trials cluster well below no-LoRA ⇒ LoRA is hurting on
  this track and base.
* If one base sits clearly above the others throughout ⇒ pick that
  base as the headline checkpoint.
* If test_metric falls while train_metric keeps climbing ⇒ overfit
  starting; check the gap plot.

In [ ]:
sink.save(plot_loss_overlay(TRACK), '06_loss_overlay')

In [ ]:
sink.save(plot_metric_overlay(TRACK, split='test'), '07_metric_overlay')

In [ ]:
sink.save(plot_metric_overlay(TRACK, split='train'), '08_metric_overlay')

In [ ]:
sink.save(plot_overfitting_diagnostic(TRACK), '09_overfitting_diagnostic')

## 4 · Final-metric comparisons by hyperparameter

**`best_test_metric`** is the best test-set score the trial
achieved during training (max of `test_metric` for PD,
min for LGD).

**`final_test_metric`** is the last-epoch value (i.e. what the
checkpoint actually delivers on disk).

Swap the `metric=` kwarg below to view the latter instead.

In [ ]:
sink.save(plot_final_metric_bar(TRACK, metric='best_test_metric'), '10_final_metric_bar')

In [ ]:
sink.save(plot_metric_heatmap(TRACK, metric='best_test_metric'), '11_metric_heatmap')

In [ ]:
sink.save(plot_lr_effect(TRACK, metric='best_test_metric'), '12_lr_effect')

In [ ]:
sink.save(plot_lora_effect(TRACK, metric='best_test_metric'), '13_lora_effect')

In [ ]:
sink.save(plot_base_ranking(TRACK, metric='best_test_metric'), '14_base_ranking')

## 5 · Time / accuracy trade-off

The Pareto-frontier-style scatter pairs **total training time** (x,
minutes) with the **final-epoch test metric** (y). A trial that
matches another's metric in less time pareto-dominates it.

The per-epoch bar plot underneath flags trials whose mean
seconds/epoch is unusually high — useful when one dataset's
chunk drags the whole epoch budget.

In [ ]:
sink.save(plot_pareto_time_vs_metric(TRACK, metric='best_test_metric'), '15_pareto_time_vs_metric')

In [ ]:
sink.save(plot_epoch_time_overlay(TRACK), '16_epoch_time_overlay')

## 6 · Convergence diagnostics

Where in the training budget does the best test metric appear?

* Most mass at < 30 % ⇒ early stop is leaving training cycles on
  the floor; consider shorter epochs or a stricter LR schedule.
* Most mass at > 80 % ⇒ trials are still improving when training
  ends; increase `cfg.train.epochs` before re-launching the sweep.

In [ ]:
sink.save(plot_convergence_speed(TRACK), '17_convergence_speed')

## 7 · Failures and leaderboard

`failed_trials()` surfaces any FAIL rows from the manifest with
their error string — most often a `CUDA OOM` from a too-large
`n_finetune_ctx_plus_query_samples`. The leaderboard at the end
is the canonical sorted view used to pick the headline
checkpoint for eval.

In [ ]:
failures = failed_trials(TRACK)
if failures.empty:
    print('No failed trials. ✔')
else:
    display(failures)

In [ ]:
trial_leaderboard(TRACK)